In [ ]:
import pandas as pd
import scanpy as sc
import scib_rapids
from scib_rapids.nearest_neighbors import NeighborsResults, pynndescent
import os, pickle, traceback
import scib
import jax
jax.config.update("jax_compilation_cache_dir", "../jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
import pickle
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9" 
i = 0

In [ ]:
import scanpy as sc

In [ ]:
a = sc.read_h5ad('data/04_integrated_data/04.17_UINMF_BBKNN.h5ad')
a

In [ ]:
def cached(method, metric, fn):
    """Run fn() once; cache to disk; skip on restart; capture errors instead of crashing."""
    path = os.path.join('data/05_evaluation', f"{method}_{metric}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)               # already done -> skip
    try:
        result = fn()
    except Exception:
        result = {"__error__": traceback.format_exc()}
        print(f"  ! {method}/{metric} FAILED (saved error, run continues)")
    with open(path, "wb") as f:
        pickle.dump(result, f)
    return result

In [ ]:
for method in methods:
    embed = embed_keys[method]
    ad = sc.read_h5ad(paths[method])
    ad = ad[ad.obs['ds_label'].notna()].copy()
    if method not in GRAPH_ONLY:
        sc.pp.neighbors(ad, use_rep = embed)
    if method == 'unintegrated':
        unintegrated = ad.copy()

    if method not in GRAPH_ONLY:
        cached(method, "silhouette_batch", 
        lambda: scib.metrics.silhouette_batch(
            ad, batch_key = 'orig.ident', label_key = 'ds_label',
            embed = embed, metric = 'euclidean', return_all = True, scale = True))

        cached(method, "pcr", 
        lambda: scib.metrics.pcr_comparison(
            adata_pre=unintegrated, adata_post=ad, covariate='orig.ident',
            embed=embed, scale=True))

        cached(method, "silhouette", 
        lambda: scib.metrics.silhouette(
            adata=ad, label_key='ds_label', embed=embed, metric='euclidean', scale=True))

        cached(method, "metrics", 
               lambda: scib.metrics.metrics(
                   adata=unintegrated, adata_int=ad, batch_key='orig.ident', label_key='ds_label',
                   embed=embed, cluster_key='cluster_scIB',
                   cluster_nmi=f"data/05_evaluation/{method}_nmi_opt_cluster.csv", type_='knn',
                   isolated_labels_asw_=False, silhouette_=False, hvg_score_=False,
                   graph_conn_=True, pcr_=False, isolated_labels_f1_=True,
                   nmi_=True, ari_=True, kBET_=True, ilisi_=True, clisi_=True,
                   cell_cycle_=False, trajectory_=False))

# scib-metrics

In [ ]:
paths = {
    'unintegrated': 'data/03_concatenated_data.h5ad',
    'fastmnn': 'data/04_integrated_data/04.01_FastMNN.h5ad',
    'samap': 'data/04_integrated_data/04.02_SAMap.h5ad',
    'cca': 'data/04_integrated_data/04.03_CCA.h5ad',
    'harmony': 'data/04_integrated_data/04.04_Harmony.h5ad',
    'scanorama': 'data/04_integrated_data/04.05_Scanorama.h5ad',
    'rpca': 'data/04_integrated_data/04.06_RPCA.h5ad',
    'scvi': 'data/04_integrated_data/04.07_scVI.h5ad',
    'scanvi': 'data/04_integrated_data/04.08_scANVI.h5ad',
    'bbknn': 'data/04_integrated_data/04.09_bbknn.h5ad',
    'liger': 'data/04_integrated_data/04.10_LIGER.h5ad',
    'liger_uinmf': 'data/04_integrated_data/04.11_LIGER_UINMF.h5ad'
}
embed_keys = {
    'unintegrated': 'X_pca',
    'fastmnn': 'X_mnn',
    'samap': 'wPCA',
    'cca': 'X_pca',
    'harmony': 'X_harmony',
    'scanorama': 'X_x_scanorama',
    'rpca': 'X_pca',
    'scvi': 'X_x_scvi',
    'scanvi': 'X_x_scanvi',
    'bbknn': 'X_pcabbknn',
    'liger': 'X_inmfnorm',
    'liger_uinmf': 'X_inmf'
}

In [ ]:
adata = sc.read_h5ad('data/03_concatenated_data.h5ad')
adata = adata[adata.obs['ds_label'].notna()].copy()
adata.obsm['X_unintegrated'] = adata.obsm['X_pca']

adata.obs['ds_label']   = adata.obs['ds_label'].astype(str).astype('category')
adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str).astype('category')

In [ ]:
adata

## Unintegrated data

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = ['X_unintegrated'],
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()

In [ ]:
bm.plot_results_table()
with open("data/05_evaluation/05_00_unintegrated_evaluated.pkl", "wb") as f:
    pickle.dump(bm, f)

## FastMNN

In [ ]:
i += 1
method = 'fastmnn'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()

In [ ]:
bm.benchmark()

In [ ]:
bm.get_results()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05.0{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

## SAMap

In [ ]:
method = 'samap'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate.obs['orig.ident'] = integrate.obs['orig.ident'].replace({'hs': 'hsapiens',
                                                                   'mm': 'mmusculus'})
integrate.obs_names = integrate.obs['orig.ident'].astype('str') + '_' + integrate.obs_names.astype('str')
integrate.obs
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

## CCA

In [ ]:
i += 1
method = 'cca'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## Harmony

In [ ]:
i += 1
method = 'harmony'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## Scanorama

In [ ]:
i += 1
method = 'scanorama'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## RPCA

In [ ]:
i += 1
method = 'rpca'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## scVI

In [ ]:
i += 1
method = 'scvi'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## scANVI

In [ ]:
i += 1
method = 'scanvi'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## BBKNN

In [ ]:
i += 1
method = 'bbknn'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
integrate

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## LIGER

In [ ]:
i += 1
method = 'liger'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05_{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

## LIGER UINMF

In [ ]:
i = 11
method = 'liger_uinmf'
embed = embed_keys[method]
integrate = sc.read_h5ad(paths[method])
integrate = integrate[integrate.obs['ds_label'].notna()].copy()
integrate = integrate[adata.obs_names].copy()
assert (integrate.obs_names == adata.obs_names).all()
new_embed = f'X_{method}'
adata.obsm[new_embed] = integrate.obsm[embed]

In [ ]:
bm = Benchmarker(
    adata,
    batch_key = 'orig.ident',
    label_key = 'ds_label',
    bio_conservation_metrics = BioConservation(),
    batch_correction_metrics = BatchCorrection(),
    embedding_obsm_keys = [new_embed],
    pre_integrated_embedding_obsm_key = 'X_unintegrated',
    n_jobs = 6
)

In [ ]:
bm.prepare()
bm.benchmark()
if i >= 10:
    i_str = f'{i}'
else:
    i_str = f'0{i}'
fname = f'data/05_evaluation/05.{i_str}_{method}_evaluated.pkl'
with open(fname, "wb") as f:
    pickle.dump(bm, f)

In [ ]:
bm.plot_results_table()

# Draft 2

In [ ]:
import scanpy as sc
import pandas as pd
import scib

In [ ]:
BATCH = 'orig.ident'
LABEL = 'ds_label'
UNINT = sc.read_h5ad('data/03_concatenated_data.h5ad')

In [ ]:
def score(adata_int, method, output_type, embed = None):
    """
    Score 1 integration result with scib
    output_type: 'embed' for embedding methods, 'knn' for graph-only methods
    """
    adata_int = adata_int.copy()
    assert (adata_int.obs_names == UNINT.obs_names).all(), 'cells must match and be in the same order'

    for key in (BATCH, LABEL):
        adata_int.obs[key] = adata_int.obs[key].astype('category')

    is_embed = output_type == 'embed'

    if is_embed:
        sc.pp.neighbors(adata_int, use_rep = embed)
    else:
        # to ensure that nothing is overwritten in the process
        neighbors_before = adata_int.obsp['connectivities'].copy()

    res = scib.me.metrics(
        UNINT, adata_int,
        batch_key = BATCH,
        label_key = LABEL,
        embed = embed if is_embed else 'X_pca',
        type_ = output_type,

        # graph-based metrics: all methods
        nmi_ = True, ari_ = True,
        isolated_labels_f1_ = True,
        graph_conn_ = True,
        kBET_ = True,
        ilisi_ = True, clisi_ = True,

        # embedding-based metrics: embedding methods only
        silhouette_ = is_embed,
        isolated_labels_asw_ = is_embed,
        pcr_ = is_embed,

        # switched off
        trajectory_ = False,
        cell_cycle_ = False,
        hvg_score_ = False
    )
    if not is_embed:
        changed = abs(adata_int.obsp['connectivities'] - neighbors_before).sum()
        assert changed == 0, 'Graph was overwritten during scoring; results are invalid'
    return res.renmae(columns = {res.columns[0]: method})


In [ ]:
adata = sc.read_h5ad('data/04_integrated_data/04.02_SAMap.h5ad')
adata

In [ ]:
raw = sc.read_h5ad('data/03_concatenated_data.h5ad')
raw

In [ ]:
species = []
for i in adata.obs['orig.ident']:
    if i == 'mm':
        species.append('mmusculus')
    else:
        species.append('hsapiens')
adata.obs['orig.ident'] = species

In [ ]:
adata.obs['orig.ident']

rownames = []
for i in range(len(species)):
    s = adata.obs['orig.ident'][i]
    c = adata.obs_names[i]
    rownames.append(f'{s}_{c}')
adata.obs_names = rownames

In [ ]:
diff1 = adata.obs_names.difference(raw.obs_names)

In [ ]:
common_cells = raw.obs_names.intersection(adata.obs_names)
adata = adata[common_cells].copy()
adata


In [ ]:
human_adata = sc.read_h5ad('data/02_annotated_data/02_human_annotated.h5ad')
human_adata.X = human_adata.layers['counts']
mouse_adata = sc.read_h5ad("data/02_annotated_data/02_mouse_annotated.h5ad")
mouse_adata.X = mouse_adata.layers['counts']

In [ ]:
human_adata

In [ ]:
mouse_adata

In [ ]:
human_adata